# Grambot grammar model training (mega)
**Before the first run:** Add Input → Your Work → the earlier Grambot notebook (the version with `grambot-model.zip`). Mega continues training that model.

Settings (right panel): **Accelerator: GPU T4 x2**, **Internet: on**.
Then **Save Version → Save & Run All (Commit)**. It runs on Kaggle's servers, so you can close the laptop.

**Run 1** trains until shortly before Kaggle's 12 h limit, then saves its progress as output.
**Run 2+**: click **Add Input → Your Work → this notebook** (the latest version), then Save & Run All again.
It picks up where the last run stopped. When everything is done, the version's **Output** tab has `grambot-model.zip`.

In [ ]:
PROFILE = "mega"     # max + 1.8 M extra real pairs, starts from the trained model; ~14 h, 2 runs
BATCH = 16           # per GPU; T4 has 16 GB
SESSION_HOURS = 11.5 # Kaggle stops a run at 12 h
import time; T0 = time.time()

In [ ]:
!mkdir -p /kaggle/working/training

In [ ]:
%%writefile /kaggle/working/training/requirements.txt
torch>=2.5
transformers>=4.46,<5
datasets>=3.0
sentencepiece
accelerate>=1.0
zstandard
optimum[onnxruntime]>=1.23
onnx


In [ ]:
%%writefile /kaggle/working/training/noise.py
"""Synthetic grammar errors: turn a correct sentence into the kind of text people actually type.

Pure Python, no dependencies. `python noise.py` prints examples and runs a self-check.
Each error function takes (sentence, rng) and returns a corrupted sentence or None if it does not apply.
"""

import random
import re


def _keep_case(src, word):
    return word[:1].upper() + word[1:] if src[:1].isupper() else word


def _swap(s, rng, table):
    """Replace one word found in `table` (right -> [wrong, ...]) with a wrong variant, preserving capitalization."""
    hits = [(m, right) for right in table for m in re.finditer(rf"\b{re.escape(right)}\b", s, re.I)]
    if not hits:
        return None
    m, right = rng.choice(hits)
    wrong = rng.choice(table[right])
    return s[: m.start()] + _keep_case(m.group(0), wrong) + s[m.end():]


# Only swaps that are always wrong: a<->an, or a dropped article. "a" -> "the" can still be correct English.
ARTICLES = {"a": ["an", ""], "an": ["a", ""], "the": [""]}
PREPOSITIONS = {p: [q for q in ["in", "on", "at", "to", "for", "of", "with", "from", "by", "about"] if q != p]
                for p in ["in", "on", "at", "to", "for", "of", "with", "from", "by", "about"]}
AGREEMENT = {"is": ["are"], "are": ["is"], "was": ["were"], "were": ["was"], "has": ["have"], "have": ["has"],
             "does": ["do"], "doesn't": ["don't"], "don't": ["doesn't"], "am": ["is", "are"]}
# base: (3rd person, past, participle)
VERBS = {"go": ("goes", "went", "gone"), "come": ("comes", "came", "come"), "see": ("sees", "saw", "seen"),
         "do": ("does", "did", "done"), "eat": ("eats", "ate", "eaten"), "write": ("writes", "wrote", "written"),
         "take": ("takes", "took", "taken"), "give": ("gives", "gave", "given"), "know": ("knows", "knew", "known"),
         "begin": ("begins", "began", "begun"), "break": ("breaks", "broke", "broken"), "speak": ("speaks", "spoke", "spoken"),
         "run": ("runs", "ran", "run"), "drink": ("drinks", "drank", "drunk"), "choose": ("chooses", "chose", "chosen"),
         "make": ("makes", "made", "made"), "get": ("gets", "got", "gotten"), "think": ("thinks", "thought", "thought"),
         "buy": ("buys", "bought", "bought"), "send": ("sends", "sent", "sent"), "need": ("needs", "needed", "needed"),
         "want": ("wants", "wanted", "wanted"), "like": ("likes", "liked", "liked"), "work": ("works", "worked", "worked"),
         "use": ("uses", "used", "used"), "help": ("helps", "helped", "helped"), "look": ("looks", "looked", "looked"),
         "say": ("says", "said", "said"), "tell": ("tells", "told", "told"), "feel": ("feels", "felt", "felt")}
TENSE = {}
for base, (third, past, part) in VERBS.items():
    for form in {base, third, past, part}:
        TENSE[form] = sorted({base, third, past, part} - {form})
HOMOPHONES = {"your": ["you're"], "you're": ["your"], "its": ["it's"], "it's": ["its"], "their": ["there", "they're"],
              "there": ["their"], "they're": ["their", "there"], "to": ["too"], "too": ["to"], "than": ["then"],
              "then": ["than"], "lose": ["loose"], "affect": ["effect"], "effect": ["affect"], "whose": ["who's"],
              "who's": ["whose"], "where": ["were"], "were": ["where"], "accept": ["except"], "quite": ["quiet"]}
PLURALS = {"people": ["peoples", "person"], "children": ["childrens", "child"], "information": ["informations"],
           "advice": ["advices"], "furniture": ["furnitures"], "equipment": ["equipments"], "news": ["new"],
           "feedback": ["feedbacks"], "homework": ["homeworks"], "research": ["researches"]}
MODAL_OF = {"could have": ["could of"], "would have": ["would of"], "should have": ["should of"], "must have": ["must of"]}
FUNCTION_WORDS = ["the", "a", "an", "to", "is", "are", "of", "that", "and", "be", "was", "it", "for"]
CONTRACTIONS = ["don't", "doesn't", "didn't", "can't", "won't", "isn't", "aren't", "wasn't", "weren't", "it's",
                "that's", "what's", "there's", "I'm", "you're", "they're", "we're", "I've", "I'll", "let's", "he's",
                "she's", "couldn't", "wouldn't", "shouldn't", "haven't", "hasn't"]


def article(s, rng): return _swap(s, rng, ARTICLES)
def preposition(s, rng): return _swap(s, rng, PREPOSITIONS)
def agreement(s, rng): return _swap(s, rng, AGREEMENT)
def tense(s, rng): return _swap(s, rng, TENSE)
def homophone(s, rng): return _swap(s, rng, HOMOPHONES)
def plural(s, rng): return _swap(s, rng, PLURALS)
def modal_of(s, rng): return _swap(s, rng, MODAL_OF)


def apostrophes(s, rng):
    out = s
    for c in CONTRACTIONS:
        out = re.sub(rf"\b{re.escape(c)}\b", lambda m: m.group(0).replace("'", ""), out, flags=re.I)
    return out if out != s else None


def drop_word(s, rng):
    return _swap(s, rng, {w: [""] for w in FUNCTION_WORDS})


def duplicate_word(s, rng):
    words = s.split(" ")
    if len(words) < 4:
        return None
    i = rng.randrange(1, len(words) - 1)
    return " ".join(words[: i + 1] + [words[i]] + words[i + 1:])


def word_order(s, rng):
    words = s.split(" ")
    if len(words) < 5:
        return None
    i = rng.randrange(1, len(words) - 2)
    if not (words[i].isalpha() and words[i + 1].isalpha()):
        return None
    words[i], words[i + 1] = words[i + 1], words[i]
    return " ".join(words)


def typo(s, rng):
    words = [(m.start(), m.group(0)) for m in re.finditer(r"[a-z]{5,}", s)]
    if not words:
        return None
    at, w = rng.choice(words)
    i = rng.randrange(1, len(w) - 1)
    bad = rng.choice([w[:i] + w[i + 1:], w[:i] + w[i + 1] + w[i] + w[i + 2:], w[:i] + w[i] + w[i:]])
    return s[:at] + bad + s[at + len(w):]


def lowercase(s, rng):
    out = re.sub(r"\bI\b", "i", s) if rng.random() < 0.5 else s
    out = out[:1].lower() + out[1:]
    return out if out != s else None


def punctuation(s, rng):
    out = s
    if "," in out and rng.random() < 0.6:
        out = out.replace(", ", " ")
    if out[-1:] in ".?!" and rng.random() < 0.7:
        out = out[:-1]
    return out if out != s else None


def comparative(s, rng):
    """taller than -> more taller than, the tallest -> the most tallest"""
    m = re.search(r"\b([a-z]{3,}er) than\b", s) or re.search(r"\bthe ([a-z]{3,}est)\b", s)
    if not m or m.group(1) in {"other", "rather", "never", "over", "under", "after", "interest", "forest", "request",
                                "honest", "latest", "rest", "test", "best"}:
        return None
    word = "more " if m.group(0).endswith("than") else "most "
    at = m.start(1)
    return s[:at] + word + s[at:]


ERRORS = [article, preposition, agreement, tense, homophone, plural, modal_of, apostrophes, drop_word,
          duplicate_word, word_order, typo, lowercase, punctuation, comparative]
WEIGHTS = [8, 3, 9, 9, 10, 2, 2, 8, 5, 2, 2, 3, 6, 9, 3]  # prepositions low: a swap is sometimes still correct


def corrupt(s, rng, max_errors=3):
    """Apply 1..max_errors different errors. Returns None if none applied."""
    out = s
    n = rng.choice([1, 1, 2, 2, 3][: max_errors + 2])
    tried = set()
    while n and len(tried) < len(ERRORS):
        fn = rng.choices(ERRORS, WEIGHTS)[0]
        if fn in tried:
            continue
        tried.add(fn)
        new = fn(out, rng)
        if new and new != out:
            out = re.sub(r" {2,}", " ", new).strip()
            n -= 1
    return out if out != s and out else None


def run_on(sentences, rng):
    """Glue 2-3 correct sentences together the way people type in chat."""
    parts = []
    for i, s in enumerate(sentences):
        p = s[:-1] if rng.random() < 0.9 else s
        if i > 0 and not re.match(r"I\b", p):
            p = p[:1].lower() + p[1:]
        parts.append(p)
    joiner = rng.choice([" ", " ", " and ", ", "])
    src = joiner.join(parts)
    return src, " ".join(sentences)


SUBJECTS = [("she", "She"), ("he", "He"), ("they", "They"), ("my mom", "My mom"), ("the teacher", "The teacher"),
            ("my friend", "My friend"), ("I", "I"), ("he just", "He just"), ("my boss", "My boss"), ("someone", "Someone"),
            ("the kid", "The kid"), ("she literally", "She literally"), ("my dad", "My dad"), ("he then", "He then")]
SPEECH_VERBS = ["said", "asked", "told me", "kept saying", "yelled", "replied", "whispered", "texted me"]
QUESTIONS = ["what's all the fuss about", "why are you so late", "where did you put my keys", "what is going on here",
             "how did you do that", "who told you that", "when are we leaving", "can you help me with this",
             "are you coming tonight", "is this your bag", "do you even care", "what time is it",
             "why is everyone so upset", "have you finished your homework", "what's wrong with you", "did you eat",
             "where are you going", "how much does it cost", "can I come too", "what's for dinner",
             "is it raining outside", "why didn't you call me", "who's going to pay for this", "are we there yet"]
STATEMENTS = ["I'm not coming tonight", "this is the best day ever", "we need to talk", "the food was amazing",
              "don't touch my phone", "you're going to love it", "I can't believe it", "it's not my fault",
              "I'll be there in five minutes", "that's not fair", "we're out of milk", "I already told you",
              "you have to try this", "it doesn't matter anymore", "the meeting is cancelled", "I love this song"]


def speech(rng):
    """She said what's all the fuss about  ->  She said, "What's all the fuss about?\""""
    low, cap = rng.choice(SUBJECTS)
    verb = rng.choice(SPEECH_VERBS)
    if rng.random() < 0.6:
        q, end = rng.choice(QUESTIONS), "?"
    else:
        q, end = rng.choice(STATEMENTS), rng.choice([".", ".", "!"])
    quoted = q[:1].upper() + q[1:] + end
    typed = apostrophes(q, rng) or q if rng.random() < 0.4 else q
    src = f"{low if rng.random() < 0.6 else cap} {verb} {typed}"
    if rng.random() < 0.2:
        src += end
    return src, f'{cap} {verb}, "{quoted}"'


REPORTED = ["what the fuss was about", "where I was going", "why I was so late", "what time it was", "who had told me that",
            "if I was coming home tonight", "whether we had finished", "how much it cost", "what was wrong with me",
            "where she had put the keys", "if I could help him", "what we were having for dinner", "why nobody called",
            "how I did that", "when we were leaving", "if it was raining outside", "what the plan was", "who was paying"]


def reported(rng):
    """Indirect speech stays unquoted: "She asked what the fuss was about." is already correct.
    Half are returned untouched, half typed chat-style and fixed only in case and full stop."""
    low, cap = rng.choice(SUBJECTS)
    verb = rng.choice(["asked", "asked me", "wanted to know", "wondered", "asked us"])
    clause = rng.choice(REPORTED)
    if low == "I" and "I " in clause:
        clause = clause.replace("I ", "she ", 1)
    good = f"{cap} {verb} {clause}."
    if rng.random() < 0.5:
        return good, good
    return f"{low} {verb} {clause}", good


if __name__ == "__main__":
    rng = random.Random(1)
    samples = ["She goes to school every day.", "I have seen that movie before.", "It's not your fault, and you know it.",
               "They were happy with the information we sent.", "He could have told me about the meeting.",
               "My brother is taller than me."]
    for fn in ERRORS:
        outs = [o for o in (fn(s, rng) for s in samples) if o]
        assert outs, f"{fn.__name__} never applied"
        print(f"{fn.__name__:15} {outs[0]}")
    for _ in range(3):
        print("corrupt        ", corrupt(rng.choice(samples), rng))
    print("run_on         ", run_on(samples[:2], rng))
    print("speech         ", speech(rng))
    print("reported       ", reported(rng))
    assert comparative("She is the tallest girl.", rng) == "She is the most tallest girl."
    assert '"' not in "".join(reported(rng)[1] for _ in range(50))
    assert all(corrupt(s, rng) != s for s in samples)
    print("self-check OK")


In [ ]:
%%writefile /kaggle/working/training/prepare_data.py
"""Build the Grambot grammar-correction training set.

Sources (all allow redistribution of a derived model):
  * agentlans/grammar-correction: a 125k-pair filtered subset of Google's C4_200M
    synthetic GEC corpus (CC BY 4.0, text from C4 / ODC-BY). Pairs whose "fix" rewrites
    the sentence are dropped, so the model learns to correct, not paraphrase.
  * Synthetic errors from noise.py on the clean side of that corpus: 15 error types
    (agreement, tense, articles, homophones, apostrophes, run-ons, punctuation, ...).
  * Reported speech that should be in quotation marks.
  * Identity pairs (correct in, correct out) so the model leaves good text alone.

Outputs in data/:
  train.jsonl   everything, for stage 1
  polish.jsonl  chat-style errors + identity, for stage 2 (short, low learning rate)
  valid.jsonl   held out, never trained on
"""

import argparse
import io
import json
import random
from pathlib import Path

from noise import corrupt, reported, run_on, speech

OUT = Path(__file__).parent / "data"

# How much synthetic data per clean sentence and how big each part is.
PROFILES = {
    "fast":   {"variants": 1, "run_ons": 15_000, "speech": 4_000, "reported": 3_000, "identity": 15_000, "polish": 30_000},
    "strong": {"variants": 1, "run_ons": 30_000, "speech": 8_000, "reported": 6_000, "identity": 30_000, "polish": 50_000},
    "max":    {"variants": 2, "run_ons": 50_000, "speech": 12_000, "reported": 10_000, "identity": 50_000, "polish": 70_000},
    # max plus ~1.8 M extra real pairs from the full C4_200M (3 shards downloaded, ~1.1 GB)
    "mega":   {"variants": 2, "run_ons": 60_000, "speech": 15_000, "reported": 20_000, "identity": 80_000, "polish": 90_000,
               "big_shards": 3, "big": 1_800_000},
}


def read_zst_jsonl(path):
    import zstandard
    with open(path, "rb") as fh:
        text = io.TextIOWrapper(zstandard.ZstdDecompressor().stream_reader(fh), encoding="utf-8")
        for line in text:
            if line.strip():
                row = json.loads(line)
                yield row["input"].strip(), row["output"].strip()


def close(a, b, min_dice=0.75):
    """Keep corrections, drop C4 pairs where the "fix" rewrites the sentence (it teaches meaning drift)."""
    wa, wb = a.lower().split(), b.lower().split()
    common = len(set(wa) & set(wb))
    return common * 2 / (len(set(wa)) + len(set(wb)) or 1) >= min_dice and abs(len(wa) - len(wb)) <= 4


def tidy(s):
    return s.replace("’", "'").replace("‘", "'").replace("“", '"').replace("”", '"').strip()


def good_target(b, vocab):
    """Raw C4_200M targets are sometimes worse than the input (typos, web junk). Keep only clean sentences
    whose every word is common in the corpus, so the model never learns to "fix" text into a typo."""
    words = b.split()
    if not (4 <= len(words) <= 40 and b.isascii() and b[:1].isupper() and b[-1:] in ".?!"):
        return False
    if " n't" in b or " 's " in b or "  " in b or "http" in b:
        return False
    return all(vocab.get(w, 0) >= 30 for w in (t.strip(".,!?;:\"'()").lower() for t in words) if w.isalpha())


def big_c4(shards, cap, skip):
    """Real pairs from the full C4_200M (liweili/c4_200m parquet mirror, 1.83 M pairs per shard)."""
    import pyarrow.parquet as pq
    from collections import Counter
    from huggingface_hub import hf_hub_download
    rows = []
    for i in range(shards):
        path = hf_hub_download("liweili/c4_200m", f"default/partial-train/{i:04d}.parquet", repo_type="dataset",
                               revision="refs/convert/parquet")
        t = pq.read_table(path, columns=["input", "output"])
        rows += zip(map(tidy, t.column("input").to_pylist()), map(tidy, t.column("output").to_pylist()))
        print(f"shard {i}: {len(rows):,} raw pairs", flush=True)
    vocab = Counter(w.strip(".,!?;:\"'()").lower() for _, b in rows for w in b.split())
    kept = [(a, b) for a, b in rows if b not in skip and close(a, b, 0.8) and good_target(b, vocab)]
    print(f"big C4_200M: kept {len(kept):,} of {len(rows):,} ({len(kept) / len(rows):.0%})", flush=True)
    return kept[:cap]


def usable(s):
    return 3 <= len(s.split()) <= 40 and s.isascii() and s[-1:] in ".?!" and '"' not in s and s[:1].isupper()


def synthetic(clean, rng, p):
    rows = []
    for s in clean:
        for _ in range(p["variants"]):
            noisy = corrupt(s, rng)
            if noisy:
                rows.append((noisy, s))
    short = [s for s in clean if len(s.split()) <= 14]
    for _ in range(p["run_ons"]):
        rows.append(run_on(rng.sample(short, rng.choice([2, 2, 3])), rng))
    rows += [speech(rng) for _ in range(p["speech"])]
    rows += [reported(rng) for _ in range(p["reported"])]
    rows += [(s, s) for s in rng.sample(clean, min(p["identity"], len(clean)))]
    return rows


def write(name, rows):
    with open(OUT / f"{name}.jsonl", "w") as fh:
        for a, b in rows:
            fh.write(json.dumps({"input": a, "output": b}) + "\n")
    print(f"{name:7} {len(rows):>9,} pairs -> {OUT / (name + '.jsonl')}")


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--profile", choices=PROFILES, default="strong")
    ap.add_argument("--seed", type=int, default=13)
    args = ap.parse_args()
    p = PROFILES[args.profile]
    rng = random.Random(args.seed)
    OUT.mkdir(exist_ok=True)

    from huggingface_hub import hf_hub_download
    repo = "agentlans/grammar-correction"
    train_src = list(read_zst_jsonl(hf_hub_download(repo, "train.jsonl.zst", repo_type="dataset")))
    valid_src = list(read_zst_jsonl(hf_hub_download(repo, "validation.jsonl.zst", repo_type="dataset")))
    print(f"downloaded {len(train_src):,} + {len(valid_src):,} C4_200M pairs")

    # Validation: real C4 pairs plus a little of every synthetic kind, all from held-out sentences.
    valid_c4 = [(a, b) for a, b in valid_src[:3_000] if close(a, b)]
    valid_clean = [b for _, b in valid_src[3_000:6_000] if usable(b)]
    small = {"variants": 1, "run_ons": 300, "speech": 150, "reported": 150, "identity": 300}
    valid = valid_c4[:1_500] + synthetic(valid_clean, rng, small)
    rng.shuffle(valid)

    # Clean sentences for synthetic training data come from train and the unused part of validation.
    c4 = [(a, b) for a, b in train_src if close(a, b)]
    if p.get("big"):
        c4 += big_c4(p["big_shards"], p["big"], skip={b for _, b in valid_src})
    # sorted: a set's order changes between runs, and resumed training needs identical data
    clean = sorted({b for _, b in train_src if usable(b)} | {b for _, b in valid_src[6_000:] if usable(b)})
    rng.shuffle(clean)
    synth = synthetic(clean, rng, p)
    train = c4 + synth
    rng.shuffle(train)

    # Stage 2: what people type in chat boxes, plus identity, so the polish pass does not forget "leave it alone".
    chatty = [r for r in synth if len(r[0].split()) <= 30]
    polish = rng.sample(chatty, min(p["polish"], len(chatty))) + rng.sample(c4, min(p["polish"] // 4, len(c4)))
    rng.shuffle(polish)

    write("train", train)
    write("polish", polish)
    write("valid", valid)
    print("example:", json.dumps(train[0]))


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /kaggle/working/training/train.py
"""Fine-tune FLAN-T5 (Apache-2.0) into Grambot's on-device grammar model.

Two stages:
  1. --stage main     all data (C4_200M + synthetic), the long run
  2. --stage polish   short pass on chat-style errors at a low learning rate, starting from stage 1

Runs on Apple Silicon (MPS), CUDA or CPU. Profiles are tuned for an 8 GB M-series Mac.
Interrupted? Run the same command again with --resume.
"""

import argparse
import json
import sys
import time
from pathlib import Path

import torch
from datasets import load_dataset
from transformers import (AutoModelForSeq2SeqLM, AutoTokenizer, DataCollatorForSeq2Seq, EarlyStoppingCallback,
                          Seq2SeqTrainer, Seq2SeqTrainingArguments, TrainerCallback)

HERE = Path(__file__).parent
PREFIX = "grammar: "  # the extension adds the same prefix

PROFILES = {
    # base model, micro-batch, epochs, learning rate. Effective batch is always 32.
    "fast":   {"base": "google/flan-t5-small", "batch": 8, "epochs": 1, "lr": 1e-3},
    "strong": {"base": "google/flan-t5-base",  "batch": 4, "epochs": 1, "lr": 5e-4},
    "max":    {"base": "google/flan-t5-base",  "batch": 4, "epochs": 2, "lr": 5e-4},
    # continues from an already trained Grambot model in init/ (Kaggle: the previous notebook's output)
    "mega":   {"base": "google/flan-t5-base",  "batch": 4, "epochs": 1, "lr": 3e-4, "init": "init/grambot-gec"},
}
EFFECTIVE_BATCH = 32
OUT_OF_TIME = 3  # exit code: stopped by --hours, run again with --resume


class Progress(TrainerCallback):
    """One plain line per log step with an ETA (progress bars do not show in cloud logs)."""

    def on_train_begin(self, args, state, control, **kwargs):
        self.t0, self.s0 = time.time(), state.global_step

    def on_log(self, args, state, control, logs=None, **kwargs):
        done = state.global_step - self.s0
        if done <= 0:
            return
        per = (time.time() - self.t0) / done
        eta = (state.max_steps - state.global_step) * per / 3600
        print(f"step {state.global_step}/{state.max_steps} | {per:.2f} s/step | ETA {eta:.1f} h | "
              + " ".join(f"{k} {v:.4g}" for k, v in (logs or {}).items() if isinstance(v, float)), flush=True)


class TimeLimit(TrainerCallback):
    """Save a checkpoint and stop cleanly before a cloud session (e.g. Kaggle's 12 h) ends."""

    def __init__(self, hours):
        self.deadline = time.time() + hours * 3600
        self.hit = False

    def on_step_end(self, args, state, control, **kwargs):
        if time.time() > self.deadline:
            self.hit = control.should_save = control.should_training_stop = True


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--profile", choices=PROFILES, default="strong")
    ap.add_argument("--stage", choices=["main", "polish"], default="main")
    ap.add_argument("--resume", action="store_true", help="continue from the last checkpoint of this stage")
    ap.add_argument("--max-train", type=int, default=0, help="limit pairs (0 = all); 300 for a smoke test")
    ap.add_argument("--max-steps", type=int, default=-1)
    ap.add_argument("--batch", type=int, default=0, help="override the micro-batch (lower it if memory runs out)")
    ap.add_argument("--hours", type=float, default=0, help="stop and checkpoint after this long (0 = no limit)")
    args = ap.parse_args()
    p = dict(PROFILES[args.profile])
    if args.batch:
        p["batch"] = args.batch

    main_out = HERE / "out" / "main"
    final_out = HERE / "out" / "grambot-gec"
    if args.stage == "main":
        start, data_file, lr, epochs, out = p["base"], "train.jsonl", p["lr"], p["epochs"], main_out
        if p.get("init"):
            start = str(HERE / p["init"])
            if not (Path(start) / "config.json").exists():
                raise SystemExit(f"Profile {args.profile} starts from a trained model, but {start} is missing.")
    else:
        if not main_out.exists():
            raise SystemExit("Run stage 1 first: python train.py --profile %s" % args.profile)
        start, data_file, lr, epochs, out = str(main_out), "polish.jsonl", p["lr"] / 3, 1, final_out

    tok = AutoTokenizer.from_pretrained(start)
    model = AutoModelForSeq2SeqLM.from_pretrained(start)
    device = "mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu"
    if p["base"].endswith("base") and device != "cuda":
        model.gradient_checkpointing_enable()  # trades ~25% speed for the memory an 8 GB Mac does not have
        model.config.use_cache = False

    data = load_dataset("json", data_files={"train": str(HERE / "data" / data_file), "valid": str(HERE / "data/valid.jsonl")})
    if args.max_train:
        data["train"] = data["train"].select(range(min(args.max_train, len(data["train"]))))
    data["valid"] = data["valid"].shuffle(seed=1).select(range(min(1_000, len(data["valid"]))))

    def encode(batch):
        enc = tok([PREFIX + s for s in batch["input"]], max_length=96, truncation=True)
        enc["labels"] = tok(text_target=batch["output"], max_length=96, truncation=True)["input_ids"]
        return enc

    data = data.map(encode, batched=True, remove_columns=["input", "output"], num_proc=4)
    steps = len(data["train"]) * epochs // EFFECTIVE_BATCH
    print(f"stage {args.stage} | {start} | device {device} | {len(data['train']):,} pairs | ~{steps:,} steps")

    smoke = args.max_steps > 0
    every = max(1, args.max_steps) if smoke else 1_000
    targs = Seq2SeqTrainingArguments(
        output_dir=str(HERE / "out" / f"checkpoints-{args.stage}"),
        per_device_train_batch_size=p["batch"],
        per_device_eval_batch_size=p["batch"] * 2,
        gradient_accumulation_steps=EFFECTIVE_BATCH // p["batch"],
        group_by_length=True,  # similar lengths per batch: less padding, less memory
        optim="adafactor",  # T5's own optimizer: far less memory than AdamW
        learning_rate=lr,
        num_train_epochs=epochs,
        max_steps=args.max_steps,
        lr_scheduler_type="cosine",
        warmup_ratio=0.02,
        label_smoothing_factor=0.1,
        eval_strategy="steps",
        eval_steps=every,
        save_steps=every,
        save_total_limit=3,
        load_best_model_at_end=True,  # keep the checkpoint with the lowest validation loss, not just the last one
        metric_for_best_model="eval_loss",
        logging_steps=1 if smoke else 50,
        bf16=device == "cuda" and torch.cuda.is_bf16_supported(),
        dataloader_pin_memory=device == "cuda",
        report_to=[],
        disable_tqdm=not sys.stdout.isatty(),
    )
    trainer = Seq2SeqTrainer(
        model=model, args=targs, train_dataset=data["train"], eval_dataset=data["valid"],
        data_collator=DataCollatorForSeq2Seq(tok, model=model, pad_to_multiple_of=32), processing_class=tok,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=4), Progress()],
    )
    limit = TimeLimit(args.hours) if args.hours else None
    if limit:
        trainer.add_callback(limit)
    trainer.train(resume_from_checkpoint=args.resume or None)
    if limit and limit.hit:
        print(f"time limit reached; checkpoint saved. Continue with: python train.py --profile {args.profile} "
              f"--stage {args.stage} --resume")
        raise SystemExit(OUT_OF_TIME)
    model.config.use_cache = True
    trainer.save_model(out)
    tok.save_pretrained(out)
    json.dump({"base": p["base"], "profile": args.profile, "stage": args.stage, "prefix": PREFIX},
              open(out / "grambot.json", "w"))
    print(f"saved -> {out}")
    if args.stage == "main":
        print(f"next: python train.py --profile {args.profile} --stage polish")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /kaggle/working/training/evaluate.py
"""Score a model the way grammar checkers are scored: per edit, not per sentence.

  precision  of the changes the model made, how many were right (false alarms hurt this)
  recall     of the changes needed, how many the model made
  F0.5       the standard GEC score; weighs precision twice as much as recall,
             because a wrong suggestion is worse than a missed one

Reports on the hand-written cases (eval_cases.json) and on held-out validation data.
Exit code 1 if the gate fails. --compare scores a second model side by side.
"""

import argparse
import json
import time
from difflib import SequenceMatcher
from pathlib import Path

HERE = Path(__file__).parent
PREFIX = "grammar: "
GATE = {"cases_f05": 0.60, "valid_f05": 0.45, "kept": 0.95}


def norm(s):
    return " ".join(s.replace("“", '"').replace("”", '"').replace("’", "'").split())


def tokens(s):
    import re
    return re.findall(r"\w+(?:'\w+)?|[^\w\s]", norm(s))


def edits(src, tgt):
    a, b = tokens(src), tokens(tgt)
    return {(i1, i2, " ".join(b[j1:j2])) for op, i1, i2, j1, j2 in SequenceMatcher(None, a, b, autojunk=False).get_opcodes()
            if op != "equal"}


class Score:
    def __init__(self):
        self.tp = self.fp = self.fn = 0

    def add(self, src, hyp, refs):
        h = edits(src, hyp)
        best = None
        for r in refs:  # credit the closest acceptable answer
            g = edits(src, r)
            tp = len(h & g)
            cand = (tp, len(h) - tp, len(g) - tp)
            if best is None or cand[0] - cand[1] - cand[2] > best[0] - best[1] - best[2]:
                best = cand
        self.tp += best[0]
        self.fp += best[1]
        self.fn += best[2]

    def prf(self):
        p = self.tp / (self.tp + self.fp) if self.tp + self.fp else 1.0
        r = self.tp / (self.tp + self.fn) if self.tp + self.fn else 1.0
        f = 1.25 * p * r / (0.25 * p + r) if p + r else 0.0
        return p, r, f


def load(path):
    import torch
    from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
    tok = AutoTokenizer.from_pretrained(path)
    dev = "cuda" if torch.cuda.is_available() else "cpu"
    model = AutoModelForSeq2SeqLM.from_pretrained(path).eval().to(dev)

    def fix(texts):
        ids = tok([PREFIX + t for t in texts], return_tensors="pt", padding=True, max_length=128, truncation=True).to(dev)
        with torch.no_grad():
            out = model.generate(**ids, max_new_tokens=128, num_beams=1)
        return tok.batch_decode(out, skip_special_tokens=True)
    return fix


def run(path, n_valid, verbose):
    fix = load(path)
    cases = json.load(open(HERE / "eval_cases.json"))
    cs, kept, kept_n = Score(), 0, 0
    t = time.perf_counter()
    outs = [fix([c[0]])[0] for c in cases]
    per_case_ms = (time.perf_counter() - t) / len(cases) * 1000
    for (inp, *refs), got in zip(cases, outs):
        correct_input = refs == [inp]
        hit = norm(got) in {norm(r) for r in refs}
        if correct_input:
            kept_n += 1
            kept += hit
        else:
            cs.add(inp, got, refs)
        if verbose:
            print(("OK " if hit else "XX ") + json.dumps(inp[:70]) + "\n     => " + json.dumps(got))

    vs = Score()
    if n_valid:
        rows = [json.loads(l) for l in open(HERE / "data/valid.jsonl")][:n_valid]
        for i in range(0, len(rows), 16):
            chunk = rows[i:i + 16]
            for r, got in zip(chunk, fix([r["input"] for r in chunk])):
                vs.add(r["input"], got, [r["output"]])
    return {"cases": cs.prf(), "valid": vs.prf() if n_valid else None, "kept": kept / kept_n, "kept_n": (kept, kept_n),
            "ms": per_case_ms}


def show(name, r):
    p, rc, f = r["cases"]
    line = f"{name:28} cases P {p:.0%} R {rc:.0%} F0.5 {f:.2f}"
    if r["valid"]:
        vp, vr, vf = r["valid"]
        line += f" | valid P {vp:.0%} R {vr:.0%} F0.5 {vf:.2f}"
    line += f" | correct kept {r['kept_n'][0]}/{r['kept_n'][1]} | {r['ms']:.0f} ms/sentence (PyTorch)"
    print(line)


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--model", default=str(HERE / "out" / "grambot-gec"))
    ap.add_argument("--compare", help="a second model to score side by side, e.g. out/main")
    ap.add_argument("--valid", type=int, default=800, help="validation pairs to score (0 = skip)")
    args = ap.parse_args()
    for path in filter(None, [args.model, args.compare]):
        if not (Path(path) / "config.json").exists():
            raise SystemExit(f"No trained model at {path}. Train first (README steps 4 and 5), then evaluate.")

    r = run(args.model, args.valid, verbose=True)
    print()
    show(Path(args.model).name, r)
    if args.compare:
        show(Path(args.compare).name, run(args.compare, args.valid, verbose=False))

    ok = r["cases"][2] >= GATE["cases_f05"] and r["kept"] >= GATE["kept"] and (not r["valid"] or r["valid"][2] >= GATE["valid_f05"])
    print("GATE", "PASS" if ok else "FAIL", GATE)
    raise SystemExit(0 if ok else 1)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /kaggle/working/training/export.py
"""Export the trained model to ONNX + int8 in the layout Transformers.js expects,
into models/grambot-gec at the repository root."""

import argparse
import shutil
from pathlib import Path

from onnxruntime.quantization import QuantType, quantize_dynamic
from optimum.exporters.onnx import main_export

HERE = Path(__file__).parent
DEST = HERE.parent / "models" / "grambot-gec"


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--model", default=str(HERE / "out" / "grambot-gec"))
    args = ap.parse_args()
    if not (Path(args.model) / "config.json").exists():
        raise SystemExit(f"No trained model at {args.model}. Train first (README steps 4 and 5), then export.")

    tmp = HERE / "out" / "onnx"
    shutil.rmtree(tmp, ignore_errors=True)
    main_export(args.model, output=tmp, task="text2text-generation-with-past")

    shutil.rmtree(DEST, ignore_errors=True)
    (DEST / "onnx").mkdir(parents=True)
    for name in ["config.json", "generation_config.json", "tokenizer.json", "tokenizer_config.json", "special_tokens_map.json"]:
        if (tmp / name).exists():
            shutil.copy(tmp / name, DEST / name)
    for part in ["encoder_model", "decoder_model_merged"]:
        quantize_dynamic(tmp / f"{part}.onnx", DEST / "onnx" / f"{part}_quantized.onnx", weight_type=QuantType.QUInt8,
                        extra_options={"EnableSubgraph": True})  # merged decoder keeps weights in If-branches
    shutil.copy(Path(args.model) / "grambot.json", DEST / "grambot.json")
    size = sum(p.stat().st_size for p in DEST.rglob("*") if p.is_file())
    print(f"exported -> {DEST} ({size / 1e6:.0f} MB)")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /kaggle/working/training/eval_cases.json
[
 [
  "So what up with that guy how is he that bad and let me tell you what she said she said what all those fuss about",
  "So, what's up with that guy? How is he that bad? And let me tell you what she said. She said, \"What's all that fuss about?\"",
  "So what's up with that guy? How is he that bad? Let me tell you what she said. She said, \"What's all that fuss about?\"",
  "So what's up with that guy? How is he that bad? And let me tell you what she said. She said, \"What's all that fuss about?\""
 ],
 [
  "He dont know the answer.",
  "He doesn't know the answer."
 ],
 [
  "She go to school every day.",
  "She goes to school every day."
 ],
 [
  "I has a apple.",
  "I have an apple."
 ],
 [
  "They was playing football yesterday.",
  "They were playing football yesterday."
 ],
 [
  "I am agree with you.",
  "I agree with you."
 ],
 [
  "Can you sent me the file?",
  "Can you send me the file?"
 ],
 [
  "She is more taller than me.",
  "She is taller than me.",
  "She is taller than I am."
 ],
 [
  "I have went there before.",
  "I have gone there before."
 ],
 [
  "Everyone have their own opinion.",
  "Everyone has their own opinion."
 ],
 [
  "The informations are useful.",
  "The information is useful."
 ],
 [
  "I look forward to meet you.",
  "I look forward to meeting you."
 ],
 [
  "If I would have known, I would have came.",
  "If I had known, I would have come."
 ],
 [
  "He said he will come tomorrow but he didnt.",
  "He said he would come tomorrow, but he didn't.",
  "He said he would come tomorrow but he didn't."
 ],
 [
  "what time is the meeting",
  "What time is the meeting?"
 ],
 [
  "its not my fault your late",
  "It's not my fault you're late."
 ],
 [
  "i think we should of left earlier",
  "I think we should have left earlier."
 ],
 [
  "she asked why are you so late",
  "She asked, \"Why are you so late?\""
 ],
 [
  "The meeting starts at 10 a.m. tomorrow.",
  "The meeting starts at 10 a.m. tomorrow."
 ],
 [
  "Could you review the pull request when you get a chance?",
  "Could you review the pull request when you get a chance?"
 ],
 [
  "I think the new design looks great.",
  "I think the new design looks great."
 ],
 [
  "Please run npm install before building.",
  "Please run npm install before building."
 ],
 [
  "She asked what the fuss was about.",
  "She asked what the fuss was about."
 ],
 [
  "We shipped the fix on Monday, and nobody has complained since.",
  "We shipped the fix on Monday, and nobody has complained since."
 ],
 [
  "me and my friend was going to the mall yesterday",
  "My friend and I were going to the mall yesterday.",
  "Me and my friend were going to the mall yesterday."
 ],
 [
  "can u tell me where is the station",
  "Can you tell me where the station is?"
 ],
 [
  "I didnt know that it was you're car.",
  "I didn't know that it was your car."
 ],
 [
  "their going to be late again",
  "They're going to be late again."
 ],
 [
  "She don't like when people is rude.",
  "She doesn't like it when people are rude.",
  "She doesn't like when people are rude."
 ],
 [
  "I seen him yesterday at the park.",
  "I saw him yesterday at the park."
 ],
 [
  "We was supposed to meet at 5 but he never came",
  "We were supposed to meet at 5, but he never came.",
  "We were supposed to meet at 5 but he never came."
 ],
 [
  "There is many reasons to stay.",
  "There are many reasons to stay."
 ],
 [
  "He go to the gym every morning before work.",
  "He goes to the gym every morning before work."
 ],
 [
  "my mom asked are you coming home tonight",
  "My mom asked, \"Are you coming home tonight?\""
 ],
 [
  "I want to loose some weight.",
  "I want to lose some weight."
 ],
 [
  "This is a easy question.",
  "This is an easy question."
 ],
 [
  "The children was playing in the the garden.",
  "The children were playing in the garden."
 ],
 [
  "I could of done it better",
  "I could have done it better."
 ],
 [
  "how are you doing today",
  "How are you doing today?"
 ],
 [
  "thanks for you help yesterday",
  "Thanks for your help yesterday."
 ],
 [
  "Its been a long day i am tired",
  "It's been a long day. I am tired.",
  "It's been a long day, and I am tired."
 ],
 [
  "Did you finished the report?",
  "Did you finish the report?"
 ],
 [
  "Let me know if you have any questions.",
  "Let me know if you have any questions."
 ],
 [
  "The API returns a 404 when the user is not found.",
  "The API returns a 404 when the user is not found."
 ],
 [
  "Honestly, I don't think it matters.",
  "Honestly, I don't think it matters."
 ],
 [
  "She said, \"I'll be there soon.\"",
  "She said, \"I'll be there soon.\""
 ],
 [
  "Can you summarize this article in three bullet points?",
  "Can you summarize this article in three bullet points?"
 ],
 [
  "Thanks again for your help!",
  "Thanks again for your help!"
 ],
 [
  "My flight lands at 6:45 p.m. on Friday.",
  "My flight lands at 6:45 p.m. on Friday."
 ],
 [
  "We're meeting at Sam's place after work.",
  "We're meeting at Sam's place after work."
 ]
]

In [ ]:
!pip install -q "transformers>=4.46,<5" "datasets>=3.0" sentencepiece "accelerate>=1.0" zstandard "optimum[onnxruntime]>=1.23" onnx
# Kaggle preinstalls diffusers, which breaks with the huggingface-hub that transformers 4.x needs,
# and optimum's ONNX export imports it if present. We never use it.
!pip uninstall -y -q diffusers

In [ ]:
import glob, os, shutil, subprocess, sys
os.chdir('/kaggle/working/training')
prev = glob.glob('/kaggle/input/**/grambot-state/out', recursive=True)
if prev:
    shutil.copytree(prev[0], 'out', dirs_exist_ok=True)
    print('continuing from', prev[0], os.listdir('out'))
else:
    print('fresh start')
if PROFILE == 'mega' and not os.path.exists('init/grambot-gec/config.json'):
    found = glob.glob('/kaggle/input/**/training/out/grambot-gec/config.json', recursive=True)
    zips = glob.glob('/kaggle/input/**/grambot-model.zip', recursive=True)
    if found:
        shutil.copytree(os.path.dirname(found[0]), 'init/grambot-gec')
    elif zips:
        !unzip -q -o {zips[0]} 'training/out/grambot-gec/*' -d /tmp/prev && mkdir -p init && mv /tmp/prev/training/out/grambot-gec init/
    else:
        raise SystemExit('Add Input -> Your Work -> the earlier Grambot notebook first (it has grambot-model.zip).')
    print('starting from', os.listdir('init/grambot-gec'))
!nvidia-smi --query-gpu=name,memory.total --format=csv
!python noise.py | tail -1
!python prepare_data.py --profile $PROFILE  # same seed every run, so a resumed run sees identical data

In [ ]:
def sh(cmd, ok=(0,)):
    print('$', ' '.join(cmd), flush=True)
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in proc.stdout:
        print(line, end='', flush=True)
    code = proc.wait()
    if code not in ok:
        raise RuntimeError(f'failed with exit code {code}: send this log to Claude')
    return code

# Smoke test (fresh start only, ~5 min): runs every step on a tiny slice so a bug fails now, not 10 h in.
if not prev:
    py = sys.executable
    tiny = [py, 'train.py', '--profile', PROFILE, '--batch', '4', '--max-train', '64', '--max-steps', '2']
    sh(tiny + ['--hours', '0.0001'], ok=(3,))  # time limit hit after step 1: proves the stop-and-save path
    sh(tiny + ['--resume'])                     # and that the next run continues from it
    sh([py, 'train.py', '--profile', PROFILE, '--batch', '4', '--max-train', '64', '--max-steps', '2', '--stage', 'polish'])
    sh([py, 'evaluate.py', '--valid', '16'], ok=(0, 1))  # 1 = gate fail, expected after 2 steps
    sh([py, 'export.py'])
    shutil.rmtree('out'); shutil.rmtree('/kaggle/working/public')
    print('SMOKE TEST PASSED', flush=True)

In [ ]:
def hours_left():
    return SESSION_HOURS - (time.time() - T0) / 3600 - 0.75  # keep time for scoring, export and saving output

def stage(name):
    done = 'out/main/grambot.json' if name == 'main' else 'out/grambot-gec/grambot.json'
    if os.path.exists(done):
        print(name, 'already done'); return True
    if hours_left() < 0.3:
        return False
    cmd = [sys.executable, 'train.py', '--profile', PROFILE, '--batch', str(BATCH), '--stage', name,
           '--hours', f'{hours_left():.2f}'] + (['--resume'] if glob.glob(f'out/checkpoints-{name}/checkpoint-*') else [])
    return sh(cmd, ok=(0, 3)) == 0  # 3 = stopped by the time limit, continue next run

finished = stage('main') and stage('polish')
print('ALL TRAINING DONE' if finished else 'Not finished yet: progress will be saved for the next run.')

In [ ]:
if finished:
    ref = 'init/grambot-gec' if os.path.exists('init/grambot-gec') else 'out/main'  # mega: vs the model it started from
    os.system(f'python evaluate.py --compare {ref} 2>&1 | tee eval.txt')
    os.system('python export.py')
    shutil.rmtree('data', ignore_errors=True)
    for d in glob.glob('out/checkpoints-*') + ['out/onnx', 'out/main', 'init']:
        shutil.rmtree(d, ignore_errors=True)
    os.chdir('/kaggle/working')
    os.system('zip -qr grambot-model.zip models/grambot-gec training/out/grambot-gec training/eval.txt')
    shutil.rmtree('training', ignore_errors=True); shutil.rmtree('public', ignore_errors=True)
    print('Download grambot-model.zip from the Output tab.')
else:
    shutil.rmtree('data', ignore_errors=True); shutil.rmtree('init', ignore_errors=True)  # both rebuilt next run
    os.makedirs('/kaggle/working/grambot-state', exist_ok=True)
    shutil.move('out', '/kaggle/working/grambot-state/out')
    print('Saved progress. Next: Add Input -> Your Work -> this notebook (latest version), then Save & Run All again.')